In [3]:
import duckdb
import pandas as pd
import numpy as np

In [12]:
# lọc ra các dòng có tỉnh Bình Dương

duckdb.query("""
    COPY (
        SELECT * 
        FROM '../datas/vietnam_real_estate.csv' 
        WHERE lower(province_name) LIKE '%bình dương%' 
           OR lower(province_name) LIKE '%binh duong%'
    ) TO '../datas/binh_duong_real_estates.csv' (HEADER, DELIMITER ',')
""")

In [6]:
INPUT_PATH = '../datas/binh_duong_real_estates.csv'
df = pd.read_csv(INPUT_PATH)

In [4]:
# Hàm format số nguyên chuẩn Việt Nam (ngăn cách bằng dấu .)
def fmt_num(val):
    return f"{val:,}".replace(',', '.')

# Kiểm tra tổng quát dữ liệu

In [7]:
total_rows = len(df)
total_cols = len(df.columns)
mem_mb = df.memory_usage(deep=True).sum() / (1024**2)

print("=" * 65)
print("1. TỔNG QUAN CẤU TRÚC (STRUCTURE)")
print("=" * 65)
print(f"• Tổng số dòng           : {fmt_num(total_rows)}")
print(f"• Tổng số cột            : {fmt_num(total_cols)}")
print(f"• Dung lượng nạp vào RAM : {mem_mb:.2f} MB")

print("\n" + "=" * 65)
print("2. DỮ LIỆU KHUYẾT THIẾU (NULL / MISSING)")
print("=" * 65)
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0].sort_values(ascending=False)

if null_cols.empty:
    print("Không có giá trị NULL nào trong toàn bộ tập dữ liệu.")
else:
    print(f"{'TÊN CỘT':<28} {'SỐ DÒNG THIẾU':>15} {'TỶ LỆ (%)':>12}")
    print("-" * 65)
    for col, cnt in null_cols.items():
        pct = (cnt / total_rows) * 100
        print(f"{col:<28} {fmt_num(cnt):>15} {pct:>11.2f}%")

print("\n" + "=" * 65)
print("3. DỮ LIỆU TRÙNG LẶP (DUPLICATES)")
print("=" * 65)
n_dup_all = df.duplicated().sum()
pct_dup_all = (n_dup_all / total_rows) * 100
print(f"• Trùng lặp hoàn toàn (mọi cột) : {fmt_num(n_dup_all):>10} dòng ({pct_dup_all:.2f}%)")

# 2. Trùng lặp theo mã định danh (ID / URL nếu có)
id_col = next((c for c in ['id', 'url', 'link'] if c in df.columns), None)
if id_col:
    id_dup = df.duplicated(subset=[id_col]).sum()
    pct_id = (id_dup / total_rows) * 100
    print(f"• Trùng lặp theo mã '{id_col}'             : {fmt_num(id_dup):>8} dòng ({pct_id:.2f}%)")

# 3. Trùng lặp nghiệp vụ (Cùng địa chỉ/dự án + cùng giá + cùng diện tích)
core_cols = [c for c in ['district_name', 'ward_name', 'street_name', 'price', 'area', 'title'] if c in df.columns]
if len(core_cols) >= 3:
    biz_dup = df.duplicated(subset=core_cols).sum()
    pct_biz = (biz_dup / total_rows) * 100
    print(f"• Trùng nghiệp vụ (Business Duplicates)  : {fmt_num(biz_dup):>8} dòng ({pct_biz:.2f}%)")
    print(f"  (Xét theo tổ hợp: {', '.join(core_cols)})")

print("=" * 65)

1. TỔNG QUAN CẤU TRÚC (STRUCTURE)
• Tổng số dòng           : 42.783
• Tổng số cột            : 19
• Dung lượng nạp vào RAM : 88.08 MB

2. DỮ LIỆU KHUYẾT THIẾU (NULL / MISSING)
TÊN CỘT                        SỐ DÒNG THIẾU    TỶ LỆ (%)
-----------------------------------------------------------------
house_depth                           41.984       98.13%
floor_count                           37.184       86.91%
road_width                            35.255       82.40%
balcony_direction                     34.536       80.72%
house_direction                       27.772       64.91%
frontage_width                        23.811       55.66%
project_name                          23.004       53.77%
street_name                           22.874       53.47%
bathroom_count                        19.268       45.04%
bedroom_count                         18.285       42.74%
ward_name                              3.375        7.89%
price                                  3.324        7.77%

3. 

# Phân tích dữ liệu bất thường

In [8]:
print("=" * 65)
print("KIỂM TRA BẤT THƯỜNG DỮ LIỆU THEO QUY TẮC NGHIỆP VỤ")
print("=" * 65)

# 1. Giá bất thường: NULL hoặc <= 0
num_price = pd.to_numeric(df['price'], errors='coerce')
invalid_price = num_price.isna() | (num_price <= 0)
cnt_price_anomaly = invalid_price.sum()

# 2. Diện tích bất thường: NULL, <= 0 hoặc < 10 m2
num_area = pd.to_numeric(df['area'], errors='coerce')
invalid_area = num_area.isna() | (num_area <= 0) | (num_area < 10)
cnt_area_anomaly = invalid_area.sum()

# 3. Trùng lặp nghiệp vụ (Business duplicates)
core_cols = ['district_name', 'ward_name', 'street_name', 'price', 'area']
dup_biz_mask = df.duplicated(subset=core_cols, keep=False)
cnt_biz_dup = df.duplicated(subset=core_cols, keep='first').sum()

# 4. Giá trị dự án chưa xác định / để trống
cnt_proj_null = df['project_name'].isna().sum()

print(f"1. Giá không hợp lệ (<= 0 hoặc NULL)    : {fmt_num(cnt_price_anomaly)} dòng")
print(f"2. Diện tích bất thường (< 10m² / NULL)  : {fmt_num(cnt_area_anomaly)} dòng")
print(f"3. Trùng lặp nghiệp vụ (cùng vị trí, giá): {fmt_num(cnt_biz_dup)} dòng")
print(f"4. Tên dự án để trống (nhà lẻ/đất dân)  : {fmt_num(cnt_proj_null)} dòng")
print("=" * 65)

KIỂM TRA BẤT THƯỜNG DỮ LIỆU THEO QUY TẮC NGHIỆP VỤ
1. Giá không hợp lệ (<= 0 hoặc NULL)    : 3.324 dòng
2. Diện tích bất thường (< 10m² / NULL)  : 2 dòng
3. Trùng lặp nghiệp vụ (cùng vị trí, giá): 13.048 dòng
4. Tên dự án để trống (nhà lẻ/đất dân)  : 23.004 dòng
